In [ ]:
# ============================================================
# STEP 1: SU(3) IRREP REGISTRY
# ============================================================

import numpy as np

def su3_dim(p, q):
    """
    Dimension formula for SU(3) irreps (p,q):
        dim = 1/2 * (p+1)(q+1)(p+q+2)
    """
    return (p+1)*(q+1)*(p+q+2)//2

def irreps_by_weight(Nmax):
    """
    Enumerate all SU(3) irreps (p,q) with p+q <= Nmax.
    Returns list of tuples [(p,q), ...] sorted by (p+q, p).
    """
    reps = []
    for p in range(Nmax+1):
        for q in range(Nmax+1-p):
            reps.append((p, q))
    # sort by total weight, then p
    reps.sort(key=lambda R: (R[0]+R[1], R[0]))
    return reps

def irreps_by_dim(Dmax):
    """
    Enumerate all SU(3) irreps (p,q) with dim(p,q) <= Dmax.
    Returns list sorted by (dimension, p+q, p).
    """
    reps = []
    # we need a search limit; dim(p,q) grows ~ O((p+q)^3)
    # safe overestimate cutoff:
    Pmax = int((2*Dmax)**(1/3)) + 3
    for p in range(Pmax):
        for q in range(Pmax):
            d = su3_dim(p, q)
            if d <= Dmax:
                reps.append((p, q))
    reps.sort(key=lambda R: (su3_dim(R[0], R[1]), R[0]+R[1], R[0]))
    return reps

def build_irrep_registry(Nmax=None, Dmax=None):
    """
    Main entry point:
      - If Nmax is given, use weight cutoff p+q <= Nmax.
      - If Dmax is given, use dimension cutoff dim(p,q) <= Dmax.
      - If both are given, take intersection.

    Returns:
      irreps: list of (p,q)
      dim:    dict {(p,q): dimension}
    """
    if Nmax is None and Dmax is None:
        raise ValueError("Must specify Nmax or Dmax.")

    reps_N = irreps_by_weight(Nmax) if Nmax is not None else None
    reps_D = irreps_by_dim(Dmax)    if Dmax is not None else None

    if Nmax is not None and Dmax is not None:
        reps = [R for R in reps_N if R in reps_D]
    else:
        reps = reps_N if reps_N is not None else reps_D

    dim_table = {R: su3_dim(*R) for R in reps}
    return reps, dim_table


# ============================================================
# EXAMPLE USAGE
# ============================================================

# Option A: cut by weight
irreps_w, dims_w = build_irrep_registry(Nmax=3)
print("Irreps with p+q <= 3:", irreps_w)
print("Dimensions:", dims_w)

# Option B: cut by dimension
irreps_d, dims_d = build_irrep_registry(Dmax=10)
print("\nIrreps with dim <= 10:", irreps_d)
print("Dimensions:", dims_d)

Irreps with p+q <= 3: [(0, 0), (0, 1), (1, 0), (0, 2), (1, 1), (2, 0), (0, 3), (1, 2), (2, 1), (3, 0)]
Dimensions: {(0, 0): 1, (0, 1): 3, (1, 0): 3, (0, 2): 6, (1, 1): 8, (2, 0): 6, (0, 3): 10, (1, 2): 15, (2, 1): 15, (3, 0): 10}

Irreps with dim <= 10: [(0, 0), (0, 1), (1, 0), (0, 2), (2, 0), (1, 1), (0, 3), (3, 0)]
Dimensions: {(0, 0): 1, (0, 1): 3, (1, 0): 3, (0, 2): 6, (2, 0): 6, (1, 1): 8, (0, 3): 10, (3, 0): 10}


In [ ]:
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

def su3_dim(p, q):
    return (p+1)*(q+1)*(p+q+2)//2

def su3_eigenphases(U):
    vals, _ = jnp.linalg.eig(U)
    idx = jnp.argsort(jnp.angle(vals))
    vals = vals[idx]
    return vals  # z1,z2,z3

def su3_character(U, p, q, tol=1e-8):
    z = su3_eigenphases(U)
    z1, z2, z3 = z[0], z[1], z[2]

    # --- SPECIAL CASE: eigenvalues nearly equal → character = dimension ---
    if (jnp.abs(z1 - z2) < tol) and (jnp.abs(z2 - z3) < tol):
        return su3_dim(p, q) * 1.0

    # --- Weyl determinant formula ---
    n1 = p + q + 2
    n2 = q + 1

    N = jnp.array([
        [z1**n1, z2**n1, z3**n1],
        [z1**n2, z2**n2, z3**n2],
        [1.0    , 1.0    , 1.0    ]
    ], dtype=jnp.complex128)

    D = jnp.array([
        [z1**2, z2**2, z3**2],
        [z1   , z2   , z3   ],
        [1.0  , 1.0  , 1.0  ]
    ], dtype=jnp.complex128)

    num = jnp.linalg.det(N)
    den = jnp.linalg.det(D)
    return num / den

# ---------------- Test ----------------

U_id = jnp.eye(3, dtype=jnp.complex128)

for R in [(0,0),(1,0),(0,1),(1,1),(2,0),(0,2)]:
    p,q = R
    print(f"χ{R}(I) =", su3_character(U_id, p, q))

χ(0, 0)(I) = 1.0
χ(1, 0)(I) = 3.0
χ(0, 1)(I) = 3.0
χ(1, 1)(I) = 8.0
χ(2, 0)(I) = 6.0
χ(0, 2)(I) = 6.0


In [ ]:
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

# ------------------------------------------------------------
# Vandermonde factor |Δ|^2 for SU(3)
# ------------------------------------------------------------

def vandermonde_sq(a, b):
    la = jnp.exp(1j*a)
    lb = jnp.exp(1j*b)
    lc = jnp.exp(-1j*(a+b))
    return jnp.abs((la - lb)*(lb - lc)*(lc - la))**2


# ------------------------------------------------------------
# SU(3) Haar integral for class functions f(α,β)
# ------------------------------------------------------------

def su3_haar_integral(f, N=200):
    """
    Compute ∫_{SU(3)} f(U) dU for class function f,
    where f takes angle inputs f(a,b).

    N = grid resolution (N×N points).
    """

    a = jnp.linspace(0, 2*jnp.pi, N)
    b = jnp.linspace(0, 2*jnp.pi, N)
    aa, bb = jnp.meshgrid(a, b, indexing='ij')

    measure = vandermonde_sq(aa, bb)
    vals = f(aa, bb)

    da = (2*jnp.pi)/(N-1)
    db = (2*jnp.pi)/(N-1)

    # normalization: 1/(6 (2π)^2)
    norm = 1.0/(6.0*(2*jnp.pi)**2)

    return norm * jnp.sum(vals * measure) * da * db


# ------------------------------------------------------------
# Test using f(U) = 1  → Haar integral should be 1
# ------------------------------------------------------------

res = su3_haar_integral(lambda a,b: jnp.ones_like(a), N=200)
print("∫ dU =", float(res))

∫ dU = 1.0167504187604688


In [ ]:
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

# ------------------------------------------------------------
# Vandermonde factor |Δ|^2 for SU(3)
# ------------------------------------------------------------

def vandermonde_sq(a, b):
    """
    |Δ|^2 for eigenvalues e^{iα}, e^{iβ}, e^{-i(α+β)}.
    """
    la = jnp.exp(1j*a)
    lb = jnp.exp(1j*b)
    lc = jnp.exp(-1j*(a+b))
    return jnp.abs((la - lb)*(lb - lc)*(lc - la))**2


# ------------------------------------------------------------
# SU(3) Haar integral for class functions f(a,b)
# using 2D Simpson rule on an N×N grid.
# ------------------------------------------------------------

def su3_haar_integral(f, N=401):
    """
    Compute ∫_{SU(3)} f(U) dU for class function f(a,b).

    f : function f(a,b) returning array same shape as a,b
    N : grid resolution (must be odd for Simpson rule)
    """

    # grid
    a = jnp.linspace(0, 2*jnp.pi, N)
    b = jnp.linspace(0, 2*jnp.pi, N)
    aa, bb = jnp.meshgrid(a, b, indexing='ij')

    # integrand
    vals = f(aa, bb) * vandermonde_sq(aa, bb)

    # Simpson weights
    w = jnp.ones(N)
    w = w.at[1:N-1:2].set(4.0)
    w = w.at[2:N-1:2].set(2.0)
    W = jnp.outer(w, w)   # 2D weight matrix

    # grid spacing
    da = (2*jnp.pi) / (N - 1)
    db = (2*jnp.pi) / (N - 1)

    # Haar normalization for SU(3)
    # ∫ dU = 1 / ( 6 (2π)^2 ) * ∬ |Δ|^2 dα dβ
    norm = 1.0 / (6.0 * (2*jnp.pi)**2)

    # Simpson factor /9: 1D Simpson is /6, but weights were 4,2,4,...1
    # Here we combine both dimensions -> (1/3)*(1/3) = 1/9
    integral = norm * jnp.sum(vals * W) * da * db / 9.0
    return integral


# ------------------------------------------------------------
# TEST: Haar integral of f = 1 SHOULD BE ≈ 1
# ------------------------------------------------------------

res = su3_haar_integral(lambda a, b: jnp.ones_like(a), N=401)
print("∫ dU =", float(res))

∫ dU = 1.0


In [ ]:
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# ---------------------- SU(3) irrep dimension ----------------------
def su3_dim(p, q):
    return (p+1)*(q+1)*(p+q+2)//2

# ---------------------- scalar SU(3) character ----------------------
def su3_character_from_eig_single(z1, z2, z3, p, q):
    n1 = p + q + 2
    n2 = q + 1

    N = jnp.array([
        [z1**n1, z2**n1, z3**n1],
        [z1**n2, z2**n2, z3**n2],
        [1.0    , 1.0    , 1.0    ]
    ], dtype=jnp.complex128)

    D = jnp.array([
        [z1**2, z2**2, z3**2],
        [z1   , z2   , z3   ],
        [1.0  , 1.0  , 1.0  ]
    ], dtype=jnp.complex128)

    return jnp.linalg.det(N) / (jnp.linalg.det(D) + 1e-14)

# ---------------------- vectorized character ----------------------
vec_su3_character_from_eig = jax.vmap(
    jax.vmap(
        su3_character_from_eig_single,
        in_axes=(0,0,0,None,None)
    ),
    in_axes=(0,0,0,None,None)
)

# ---------------------- Vandermonde ----------------------
def vandermonde_sq(a, b):
    la = jnp.exp(1j*a)
    lb = jnp.exp(1j*b)
    lc = jnp.exp(-1j*(a+b))
    return jnp.abs((la - lb)*(lb - lc)*(lc - la))**2

# ---------------------- Haar integration ----------------------
def su3_haar_integral(f, N=401):
    a = jnp.linspace(0, 2*jnp.pi, N)
    b = jnp.linspace(0, 2*jnp.pi, N)
    aa, bb = jnp.meshgrid(a, b, indexing='ij')

    vals = f(aa, bb) * vandermonde_sq(aa, bb)

    w = jnp.ones(N)
    w = w.at[1:N-1:2].set(4.0)
    w = w.at[2:N-1:2].set(2.0)
    W = jnp.outer(w, w)

    da = (2*jnp.pi)/(N - 1)
    db = (2*jnp.pi)/(N - 1)

    norm = 1.0/(6.0*(2*jnp.pi)**2)
    return norm * jnp.sum(vals * W) * da * db / 9.0

# ---------------------- compute c_R(beta) ----------------------
def compute_c_R_beta(p, q, beta, N=401):
    dimR = su3_dim(p, q)

    def f(a, b):
        z1 = jnp.exp(1j*a)
        z2 = jnp.exp(1j*b)
        z3 = jnp.exp(-1j*(a+b))

        chi = vec_su3_character_from_eig(z1, z2, z3, p, q)
        ReTr = jnp.real(z1 + z2 + z3)
        weight = jnp.exp((beta/3.0)*ReTr)

        return weight * jnp.conjugate(chi)

    integral = su3_haar_integral(f, N=N)
    return integral / dimR

# ---------------------- example ----------------------
beta = 1.0
irreps = [(0,0),(1,0),(0,1),(1,1),(2,0),(0,2)]

for (p,q) in irreps:
    cR = compute_c_R_beta(p, q, beta, N=401)
    print(f"c_({p},{q}) = {float(jnp.real(cR))}")

c_(0,0) = 1.0297431688041572
c_(1,0) = 0.061914909103591896
c_(0,1) = 0.06191490910359194
c_(1,1) = 0.003966723977380526
c_(2,0) = 0.00278696307834035
c_(0,2) = 0.0027869630783403467


In [ ]:
# ============================================================
# Correct SU(3) Fusion Rules via Weight Addition + Simple-root shifts
# ============================================================

import itertools

# Simple roots in Dynkin basis
alpha1 = jnp.array([2, -1])
alpha2 = jnp.array([-1, 2])

def su3_fuse(p, q, r, s):
    """
    Exact SU(3) tensor product decomposition using:
        highest weight addition +
        subtraction of simple roots α1, α2
        with dominant-weight filtering.

    Returns all irreps (P,Q) that appear in (p,q)⊗(r,s).
    """

    lam = jnp.array([p, q], dtype=int)
    mu  = jnp.array([r, s], dtype=int)

    highest = lam + mu

    irreps = set()

    # Search window for root subtractions
    # Large enough to cover all SU(3) decompositions for moderate reps
    R = 10

    for i in range(R):
        for j in range(R):
            w = highest - i * alpha1 - j * alpha2
            P = int(w[0])
            Q = int(w[1])
            # keep only dominant weights
            if P >= 0 and Q >= 0:
                irreps.add((P, Q))

    return sorted(irreps, key=lambda x: (x[0] + x[1], x[0]))


# ============================================================
# TEST CASES
# ============================================================

tests = [
    ((1,0),(1,0)),
    ((1,0),(0,1)),
    ((1,1),(1,1)),
]

for (p,q),(r,s) in tests:
    print(f"{(p,q)} ⊗ {(r,s)} → {su3_fuse(p,q,r,s)}")

(1, 0) ⊗ (1, 0) → [(0, 1), (2, 0)]
(1, 0) ⊗ (0, 1) → [(0, 0), (1, 1)]
(1, 1) ⊗ (1, 1) → [(0, 0), (1, 1), (0, 3), (3, 0), (2, 2)]


In [ ]:
# ============================================================
# STEP 6 — Local SU(3) Boltzmann Tensor  T_{ijkl}(β)
# ============================================================

import jax.numpy as jnp

def su3_dim(p, q):
    return (p+1)*(q+1)*(p+q+2)//2

# ---------- Correct SU(3) Fusion from Step 5 ----------
alpha1 = jnp.array([2, -1])
alpha2 = jnp.array([-1, 2])

def su3_fuse(p, q, r, s):
    irreps = set()
    lam = jnp.array([p, q], dtype=int)
    mu  = jnp.array([r, s], dtype=int)
    highest = lam + mu
    R = 10
    for i in range(R):
        for j in range(R):
            w = highest - i * alpha1 - j * alpha2
            P = int(w[0])
            Q = int(w[1])
            if P >= 0 and Q >= 0:
                irreps.add((P, Q))
    return sorted(irreps, key=lambda x: (x[0] + x[1], x[0]))

# ---------- Tensor constructor ----------
def build_su3_tensor(irreps, cR_table, beta):
    nR = len(irreps)
    T = jnp.zeros((nR, nR, nR, nR), dtype=jnp.float64)

    fusion_cache = {}
    for i, Ri in enumerate(irreps):
        for j, Rj in enumerate(irreps):
            fusion_cache[(i,j)] = set(su3_fuse(Ri[0], Ri[1], Rj[0], Rj[1]))

    for i in range(nR):
        for j in range(nR):
            fuse_ij = fusion_cache[(i,j)]
            for k in range(nR):
                for l in range(nR):
                    fuse_kl = fusion_cache[(k,l)]
                    common = fuse_ij.intersection(fuse_kl)
                    if common:
                        val = sum(cR_table.get(R, 0.0) for R in common)
                        T = T.at[i,j,k,l].set(val)
    return T

# ============================================================
# EXAMPLE EXECUTION WITH PRINT
# ============================================================

# Example irreps
irreps = [(0,0),(1,0),(0,1),(1,1),(2,0),(0,2)]

# Example c_R(beta) table (placeholder using dim; replace with Step 4 values)
beta = 1.0
cR_table = {(p,q): su3_dim(p,q)*0.01 for (p,q) in irreps}

# Build tensor
T = build_su3_tensor(irreps, cR_table, beta)

print("Tensor shape:", T.shape)
print("Nonzero tensor entries:", jnp.sum(T != 0.0))
print("Sample slice T[0,:,:,:] =", T[0])

Tensor shape: (6, 6, 6, 6)
Nonzero tensor entries: 432
Sample slice T[0,:,:,:] = [[[0.01 0.   0.   0.01 0.   0.  ]
  [0.   0.   0.01 0.   0.01 0.  ]
  [0.   0.01 0.   0.   0.   0.01]
  [0.01 0.   0.   0.01 0.   0.  ]
  [0.   0.01 0.   0.   0.   0.01]
  [0.   0.   0.01 0.   0.01 0.  ]]

 [[0.   0.03 0.   0.   0.   0.03]
  [0.03 0.   0.   0.03 0.   0.  ]
  [0.   0.   0.03 0.   0.03 0.  ]
  [0.   0.03 0.   0.   0.   0.03]
  [0.   0.   0.03 0.   0.03 0.  ]
  [0.03 0.   0.   0.03 0.   0.  ]]

 [[0.   0.   0.03 0.   0.03 0.  ]
  [0.   0.03 0.   0.   0.   0.03]
  [0.03 0.   0.   0.03 0.   0.  ]
  [0.   0.   0.03 0.   0.03 0.  ]
  [0.03 0.   0.   0.03 0.   0.  ]
  [0.   0.03 0.   0.   0.   0.03]]

 [[0.01 0.   0.   0.09 0.   0.  ]
  [0.   0.   0.09 0.   0.09 0.  ]
  [0.   0.09 0.   0.   0.   0.09]
  [0.09 0.   0.   0.09 0.   0.  ]
  [0.   0.09 0.   0.   0.   0.09]
  [0.   0.   0.09 0.   0.09 0.  ]]

 [[0.   0.   0.03 0.   0.09 0.  ]
  [0.   0.09 0.   0.   0.   0.09]
  [0.03 0.   0.   0.09 0.  

In [ ]:
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# ------------------------------------------------------------
# Basic HOTRG vertical contraction (correct version)
# ------------------------------------------------------------

def hotrg_merge_vertical(T, chi):
    D = T.shape[0]  # T has shape (D,D,D,D)

    # Contract down leg of T with up leg of T
    # M[l1, u1, r1, d1, l2, r2, u2]
    M = jnp.tensordot(T, T, axes=[[3],[1]])

    # Rearrange into matrix for SVD:
    # left side  = (l1, d1, l2, d2)
    # right side = (u1, r1, r2, u2)
    M = jnp.transpose(M, (0,3,4,6, 1,2,5))
    M = M.reshape(D*D*D*D, D*D*D*D)

    # SVD
    U, S, Vh = jnp.linalg.svd(M, full_matrices=False)

    chi_eff = min(chi, U.shape[1])
    U = U[:, :chi_eff]
    S = S[:chi_eff]
    Vh = Vh[:chi_eff, :]

    # Rebuild contracted matrix
    M2 = (U * S) @ Vh

    # Back to tensor
    M2 = M2.reshape(D, D, D, D, D, D, D, D)

    # Fuse vertical bonds (u1, u2)
    T_new = jnp.einsum("l d l2 d2 u r r2 u2 -> l u r d", M2)

    return T_new


# ------------------------------------------------------------
# HOTRG sweep (vertical + horizontal)
# ------------------------------------------------------------

def hotrg_sweep(T, chi, nsteps=1):
    for _ in range(nsteps):

        # vertical contraction
        T = hotrg_merge_vertical(T, chi)

        # rotate for horizontal contraction
        T = jnp.transpose(T, (1,2,3,0))
        T = hotrg_merge_vertical(T, chi)
        T = jnp.transpose(T, (3,0,1,2))

    return T


# ------------------------------------------------------------
# EXECUTION (T_su3 from Step 6 must exist)
# ------------------------------------------------------------

chi = 4
T_coarse = hotrg_sweep(T_su3, chi, nsteps=1)

print("T_su3 shape:", T_su3.shape)
print("T_coarse shape:", T_coarse.shape)
print("Nonzero entries coarse:", jnp.sum(T_coarse != 0.0))

ValueError: axis 6 is out of bounds for array of dimension 6

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_enable_x64", True)

# ============================================================
# SU(3) generators (anti-Hermitian basis)
# ============================================================

def su3_generators():
    lam = []
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]], jnp.complex128))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]], jnp.complex128))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]], jnp.complex128)/jnp.sqrt(3.0))
    lam = jnp.stack(lam, axis=0)
    return 1j * lam / 2.0  # anti-Hermitian

_T = su3_generators()

def su3_alg_from_vec(a):
    return jnp.einsum("...a,aij->...ij", a, _T)

def su3_exp(A):
    return jax.scipy.linalg.expm(A)

# ============================================================
# Lattice parametrization
# ============================================================

def n_params(L: int) -> int:
    return (L**4) * 4 * 8  # sites * directions * 8 generators

def theta_to_field(theta: jnp.ndarray, L: int) -> jnp.ndarray:
    return theta.reshape(L, L, L, L, 4, 8)

def build_links(theta: jnp.ndarray, L: int) -> jnp.ndarray:
    theta_field = theta_to_field(theta, L)
    flat = theta_field.reshape(-1, 8)
    A = jax.vmap(su3_alg_from_vec)(flat)
    U = jax.vmap(su3_exp)(A)
    return U.reshape(L, L, L, L, 4, 3, 3)

# ============================================================
# Wilson action and Hessian (JAX)
# ============================================================

def wilson_action(theta: jnp.ndarray, L: int, beta: float = 1.0) -> jnp.ndarray:
    U = build_links(theta, L)
    S = 0.0
    for mu in range(4):
        for nu in range(mu+1, 4):
            U1 = U[..., mu, :, :]
            U2 = jnp.roll(U[..., nu, :, :], -1, axis=mu)
            U3 = jnp.conjugate(jnp.swapaxes(jnp.roll(U[..., mu, :, :], -1, axis=nu), -1, -2))
            U4 = jnp.conjugate(jnp.swapaxes(U[..., nu, :, :], -1, -2))
            P = U1 @ U2 @ U3 @ U4
            trP = jnp.real(jnp.einsum("...ii->...", P))
            S = S + jnp.sum(1.0 - trP/3.0)
    return beta * S

def build_hessian(L: int, beta: float = 1.0) -> np.ndarray:
    n = n_params(L)
    def action_wrap(theta):
        return wilson_action(theta, L, beta=beta)
    grad_S = jax.grad(action_wrap)
    hess_fn = jax.jacfwd(grad_S)
    theta0 = jnp.zeros((n,), dtype=jnp.float64)
    H = hess_fn(theta0)
    H = np.array(H, dtype=float)
    H = 0.5 * (H + H.T)
    return H

# ============================================================
# Gauge generator G and projector P
# ============================================================

def _site_index(L: int, x0: int, x1: int, x2: int, x3: int) -> int:
    return ((x0 * L + x1) * L + x2) * L + x3

def _theta_index(L: int, x0: int, x1: int, x2: int, x3: int, mu: int, a: int) -> int:
    n_color = 8
    s = _site_index(L, x0, x1, x2, x3)
    return (s * 4 + mu) * n_color + a

def _alpha_index(L: int, x0: int, x1: int, x2: int, x3: int, a: int) -> int:
    n_color = 8
    s = _site_index(L, x0, x1, x2, x3)
    return s * n_color + a

def build_gauge_matrix(L: int) -> np.ndarray:
    n_color = 8
    n_sites = L**4
    n_alpha = n_sites * n_color
    n_theta = n_params(L)
    G = np.zeros((n_theta, n_alpha), dtype=float)
    for x0 in range(L):
        for x1 in range(L):
            for x2 in range(L):
                for x3 in range(L):
                    for a in range(n_color):
                        col = _alpha_index(L, x0, x1, x2, x3, a)
                        for mu in range(4):
                            row_plus = _theta_index(L, x0, x1, x2, x3, mu, a)
                            G[row_plus, col] += 1.0
                            coords_minus = [x0, x1, x2, x3]
                            coords_minus[mu] = (coords_minus[mu] - 1) % L
                            row_minus = _theta_index(L, *coords_minus, mu, a)
                            G[row_minus, col] -= 1.0
    return G

def projector_from_gauge(G: np.ndarray, tol: float = 1e-10) -> np.ndarray:
    n_theta, _ = G.shape
    U, s, _ = np.linalg.svd(G, full_matrices=False)
    mask = s > tol
    if np.any(mask):
        U_im = U[:, mask]
        P = np.eye(n_theta) - U_im @ U_im.T
    else:
        P = np.eye(n_theta)
    P = 0.5 * (P + P.T)
    return P

# ============================================================
# Physical Hessian and C_W(a) measurement
# ============================================================

def physical_hessian(L: int, beta: float = 1.0, tol: float = 1e-10):
    H = build_hessian(L, beta=beta)
    G = build_gauge_matrix(L)
    P = projector_from_gauge(G, tol=tol)
    H_phys = P @ H @ P
    H_phys = 0.5 * (H_phys + H_phys.T)
    return H_phys

def compute_CW(L: int, beta: float, tol_ev: float = 1e-6):
    H_phys = physical_hessian(L, beta=beta)
    w = np.linalg.eigvalsh(H_phys)
    # ignore near-zero gauge/toron modes
    w_phys = w[np.abs(w) > tol_ev]
    if w_phys.size == 0:
        return 0.0
    return float(w_phys.min())

# ============================================================
# SWEEP OVER a, β(a) AND COMPUTE C_W(a) AND M_eff(a) = C_W(a)/a
# ============================================================

# Example: three lattice spacings and corresponding betas.
# (You should replace beta_values with the scaling you want to probe)
a_values    = [0.25, 0.20, 0.16]       # example spacings
beta_values = [5.7, 5.9, 6.1]          # example Wilson betas (placeholder)

L = 2  # we keep L=2 fixed here for curvature measurement

results = []
for a, beta in zip(a_values, beta_values):
    CW = compute_CW(L, beta)
    Meff = CW / a
    results.append((a, beta, CW, Meff))
    print(f"a={a:.3f}, beta={beta:.3f}, C_W(a)={CW:.6e}, M_eff(a)={Meff:.6e}")

# 'results' now holds (a, beta, C_W, M_eff) for further analysis or plotting

a=0.250, beta=5.700, C_W(a)=3.800000e+00, M_eff(a)=1.520000e+01
a=0.200, beta=5.900, C_W(a)=3.933333e+00, M_eff(a)=1.966667e+01
a=0.160, beta=6.100, C_W(a)=4.066667e+00, M_eff(a)=2.541667e+01


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
jax.config.update("jax_enable_x64", True)

# ============================================================
# SU(3) generators
# ============================================================

def su3_generators():
    lam = []
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]], jnp.complex128))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]], jnp.complex128))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]], jnp.complex128)/jnp.sqrt(3.0))
    lam = jnp.stack(lam, axis=0)
    return 1j * lam / 2.0

_T = su3_generators()

def su3_alg_from_vec(a):
    return jnp.einsum("...a,aij->...ij", a, _T)

def su3_exp(A):
    return jax.scipy.linalg.expm(A)

# ============================================================
# Lattice parametrization
# ============================================================

def n_params(L: int) -> int:
    return L**4 * 4 * 8

def theta_to_field(theta, L):
    return theta.reshape(L, L, L, L, 4, 8)

def build_links(theta, L):
    field = theta_to_field(theta, L).reshape(-1, 8)
    mats = jax.vmap(su3_exp)(jax.vmap(su3_alg_from_vec)(field))
    return mats.reshape(L, L, L, L, 4, 3, 3)

# ============================================================
# Wilson action + Hessian
# ============================================================

def wilson_action(theta, L, beta=1.0):
    U = build_links(theta, L)
    S = 0.0
    for mu in range(4):
        for nu in range(mu+1, 4):
            U1 = U[..., mu, :, :]
            U2 = jnp.roll(U[..., nu, :, :], -1, axis=mu)
            U3 = jnp.conjugate(jnp.swapaxes(jnp.roll(U[..., mu, :, :], -1, axis=nu), -1, -2))
            U4 = jnp.conjugate(jnp.swapaxes(U[..., nu, :, :], -1, -2))
            P = U1 @ U2 @ U3 @ U4
            trP = jnp.real(jnp.einsum("...ii->...", P))
            S += jnp.sum(1.0 - trP/3.0)
    return beta * S

def build_hessian(L, beta=1.0):
    n = n_params(L)
    def act(theta): return wilson_action(theta, L, beta)
    gradS = jax.grad(act)
    H_fn = jax.jacfwd(gradS)
    theta0 = jnp.zeros((n,), float)
    H = np.array(H_fn(theta0))
    return 0.5*(H + H.T)

# ============================================================
# Gauge projector
# ============================================================

def _site(L,x0,x1,x2,x3):
    return ((x0*L + x1)*L + x2)*L + x3

def _tidx(L,x0,x1,x2,x3,mu,a):
    return (_site(L,x0,x1,x2,x3)*4 + mu)*8 + a

def _aidx(L,x0,x1,x2,x3,a):
    return _site(L,x0,x1,x2,x3)*8 + a

def build_gauge_matrix(L):
    n_theta = n_params(L)
    n_alpha = L**4 * 8
    G = np.zeros((n_theta, n_alpha))
    for x0 in range(L):
        for x1 in range(L):
            for x2 in range(L):
                for x3 in range(L):
                    for a in range(8):
                        col = _aidx(L,x0,x1,x2,x3,a)
                        for mu in range(4):
                            G[_tidx(L,x0,x1,x2,x3,mu,a), col] += 1.0
                            xm=[x0,x1,x2,x3]
                            xm[mu]=(xm[mu]-1)%L
                            G[_tidx(L,*xm,mu,a), col] -= 1.0
    return G

def projector_from_gauge(G):
    U,s,_=np.linalg.svd(G,full_matrices=False)
    mask=s>1e-10
    if np.any(mask):
        return np.eye(G.shape[0]) - U[:,mask] @ U[:,mask].T
    return np.eye(G.shape[0])

# ============================================================
# Physical Hessian + λ_min
# ============================================================

def physical_hessian(L, beta):
    H = build_hessian(L, beta)
    G = build_gauge_matrix(L)
    P = projector_from_gauge(G)
    H_phys = P @ H @ P
    return 0.5*(H_phys + H_phys.T)

def lambda_min_phys(H_phys, tol=1e-6):
    w = np.linalg.eigvalsh(H_phys)
    w = w[np.abs(w)>tol]
    return float(w.min()) if len(w)>0 else 0.0

# ============================================================
# Riccati flow + renormalized curvature
# ============================================================

def riccati_step(H, eta):
    I = np.eye(H.shape[0])
    return H @ np.linalg.inv(I + eta*H)

def renormalized_CW(a, L, beta, k=10, eta=0.05):
    H_phys = physical_hessian(L, beta)
    H = H_phys.copy()
    for _ in range(k):
        H = riccati_step(H, eta)
    CWr = lambda_min_phys(H)
    return CWr, CWr/a

# ============================================================
# Example sweep over lattice spacings a
# ============================================================

a_values    = [0.25, 0.20, 0.16]
beta_values = [5.7, 5.9, 6.1]   # placeholder scaling

L = 2
for a,beta in zip(a_values, beta_values):
    CWr, Meff = renormalized_CW(a, L, beta, k=12, eta=0.05)
    print(f"a={a:.3f}, beta={beta:.3f}, CWr={CWr:.6e}, M_eff={Meff:.6e}")

a=0.250, beta=5.700, CWr=1.158537e+00, M_eff=4.634146e+00
a=0.200, beta=5.900, CWr=1.170635e+00, M_eff=5.853175e+00
a=0.160, beta=6.100, CWr=1.182171e+00, M_eff=7.388566e+00


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
jax.config.update("jax_enable_x64", True)

# ============================================================
# (1) SU(3) generators ----------------------------------------
# ============================================================

def su3_generators():
    lam = []
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]], jnp.complex128))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]], jnp.complex128))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]], jnp.complex128))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]], jnp.complex128)/jnp.sqrt(3.0))
    lam = jnp.stack(lam, axis=0)
    return 1j * lam / 2.0

_T = su3_generators()

def su3_alg_from_vec(a):
    return jnp.einsum("...a,aij->...ij", a, _T)

def su3_exp(A):
    return jax.scipy.linalg.expm(A)

# ============================================================
# (2) Lattice parametrization ---------------------------------
# ============================================================

def n_params(L): return L**4 * 4 * 8

def theta_to_field(theta, L):
    return theta.reshape(L, L, L, L, 4, 8)

def build_links(theta, L):
    field = theta_to_field(theta, L).reshape(-1, 8)
    mats = jax.vmap(su3_exp)(jax.vmap(su3_alg_from_vec)(field))
    return mats.reshape(L, L, L, L, 4, 3, 3)

# ============================================================
# (3) Wilson action + Hessian ---------------------------------
# ============================================================

def wilson_action(theta, L, beta=1.0):
    U = build_links(theta, L)
    S = 0.0
    for mu in range(4):
        for nu in range(mu+1,4):
            U1 = U[..., mu,:,:]
            U2 = jnp.roll(U[..., nu,:,:], -1, axis=mu)
            U3 = jnp.conjugate(jnp.swapaxes(jnp.roll(U[..., mu,:,:], -1, axis=nu), -1,-2))
            U4 = jnp.conjugate(jnp.swapaxes(U[..., nu,:,:], -1,-2))
            P = U1 @ U2 @ U3 @ U4
            S += jnp.sum(1.0 - jnp.real(jnp.einsum("...ii->...", P))/3.0)
    return beta*S

def hessian(L, beta):
    n = n_params(L)
    def act(theta): return wilson_action(theta, L, beta)
    gradS = jax.grad(act)
    H_fn = jax.jacfwd(gradS)
    theta0 = jnp.zeros((n,), float)
    H = np.array(H_fn(theta0))
    return 0.5*(H + H.T)

# ============================================================
# (4) Gauge projector -----------------------------------------
# ============================================================

def _site(L,x0,x1,x2,x3):
    return ((x0*L + x1)*L + x2)*L + x3

def _tidx(L,x0,x1,x2,x3,mu,a):
    return (_site(L,x0,x1,x2,x3)*4 + mu)*8 + a

def _aidx(L,x0,x1,x2,x3,a):
    return _site(L,x0,x1,x2,x3)*8 + a

def gauge_matrix(L):
    n_t = n_params(L)
    n_a = L**4 * 8
    G = np.zeros((n_t, n_a))
    for x0 in range(L):
        for x1 in range(L):
            for x2 in range(L):
                for x3 in range(L):
                    for a in range(8):
                        col = _aidx(L,x0,x1,x2,x3,a)
                        for mu in range(4):
                            G[_tidx(L,x0,x1,x2,x3,mu,a), col] += 1.0
                            xm=[x0,x1,x2,x3]
                            xm[mu]=(xm[mu]-1)%L
                            G[_tidx(L,*xm,mu,a), col] -= 1.0
    return G

def projector(G):
    U,s,_=np.linalg.svd(G,full_matrices=False)
    mask=s>1e-10
    if np.any(mask):
        Uim = U[:,mask]
        P = np.eye(G.shape[0]) - Uim @ Uim.T
    else:
        P = np.eye(G.shape[0])
    return 0.5*(P+P.T)

# ============================================================
# (5) Build coarse J matrix (fine → coarse)
# ============================================================

def build_J_fine_to_coarse(L):
    Lc = L//2
    nf = n_params(L)
    nc = n_params(Lc)
    J = np.zeros((nf,nc))

    # Coarse site only has (0,0,0,0)
    for mu in range(4):
        for a in range(8):
            cidx = (0*4 + mu)*8 + a
            fine_sites=[(0,0,0,0),(1 if mu==0 else 0,
                                  1 if mu==1 else 0,
                                  1 if mu==2 else 0,
                                  1 if mu==3 else 0)]
            for (x0,x1,x2,x3) in fine_sites:
                s=_site(L,x0,x1,x2,x3)
                fidx=(s*4 + mu)*8 + a
                J[fidx,cidx]=1.0
    return J

# ============================================================
# (6) Coarse Hessian = Jᵀ H J
# ============================================================

def coarse_hessian(H_fine, P_fine, J, P_coarse):
    Hf = P_fine @ H_fine @ P_fine
    Hc = J.T @ Hf @ J
    Hc = P_coarse @ Hc @ P_coarse
    return 0.5*(Hc + Hc.T)

# ============================================================
# (7) Riccati flow + λ_min
# ============================================================

def riccati(H, k=10, eta=0.05):
    I=np.eye(H.shape[0])
    Hn=H.copy()
    for _ in range(k):
        Hn = Hn @ np.linalg.inv(I + eta*Hn)
    return 0.5*(Hn + Hn.T)

def lambda_min_phys(H, tol=1e-6):
    w=np.linalg.eigvalsh(H)
    w=w[np.abs(w)>tol]
    return float(w.min()) if len(w)>0 else 0.0

# ============================================================
# (8) EXECUTION — PRINT RESULTS
# ============================================================

L = 2        # fine lattice
Lc = 1       # coarse lattice
a = 0.2
beta = 5.9   # placeholder

# fine Hessian + projector
Hf = hessian(L, beta)
Gf = gauge_matrix(L)
Pf = projector(Gf)

# coarse projector
Gc = gauge_matrix(Lc)
Pc = projector(Gc)

# coarse map
J = build_J_fine_to_coarse(L)

# coarse physical Hessian
Hc = coarse_hessian(Hf, Pf, J, Pc)

# Riccati renormalization
HcR = riccati(Hc, k=12, eta=0.05)

# curvature constant
CWr = lambda_min_phys(HcR)
Meff = CWr / (2*a)

print("Coarse renormalized curvature CWr =", CWr)
print("M_eff(2a) =", Meff)

Coarse renormalized curvature CWr = 1.460396039603958
M_eff(2a) = 3.6509900990098947


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
jax.config.update("jax_enable_x64", True)

# ============================================================
# SU(3) generators
# ============================================================

def su3_generators():
    lam=[]
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]],complex))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]],complex)/np.sqrt(3))
    return 1j * jnp.stack(lam)/2

_T = su3_generators()

def su3_alg_from_vec(a):
    return jnp.einsum("...a,aij->...ij", a, _T)

def su3_exp(A):
    return jax.scipy.linalg.expm(A)


# ============================================================
# Lattice parametrization
# ============================================================

def n_params(L):
    return L**4 * 4 * 8

def theta_to_field(theta, L):
    return theta.reshape(L,L,L,L,4,8)

def build_links(theta, L):
    flat = theta_to_field(theta, L).reshape(-1,8)
    mats = jax.vmap(su3_exp)(jax.vmap(su3_alg_from_vec)(flat))
    return mats.reshape(L,L,L,L,4,3,3)


# ============================================================
# Wilson action & gradient
# ============================================================

def wilson_action(theta, L, beta=1.0):
    U = build_links(theta, L)
    S = 0.0
    for mu in range(4):
        for nu in range(mu+1,4):
            U1=U[...,mu,:,:]
            U2=jnp.roll(U[...,nu,:,:], -1, axis=mu)
            U3=jnp.conjugate(jnp.swapaxes(jnp.roll(U[...,mu,:,:],-1,axis=nu), -1,-2))
            U4=jnp.conjugate(jnp.swapaxes(U[...,nu,:,:], -1,-2))
            P = U1@U2@U3@U4
            S += jnp.sum(1.0 - jnp.real(jnp.einsum("...ii->...",P))/3)
    return beta*S

grad_S = jax.grad(wilson_action)


# ============================================================
# Matrix-free Hessian-vector product (fixed)
# ============================================================

def hessian_vec(theta, v, L, beta=1.0):
    theta_j = jnp.array(theta)
    v_j = jnp.array(v)
    def g(x): return grad_S(x, L, beta)
    _, Hv = jax.jvp(g, (theta_j,), (v_j,))
    return np.array(Hv)


# ============================================================
# Gauge projector for vectors
# ============================================================

def gauge_matrix(L):
    n_t=n_params(L); n_a=L**4*8
    G=np.zeros((n_t,n_a))
    def site(x0,x1,x2,x3): return ((x0*L+x1)*L+x2)*L+x3
    def tidx(x0,x1,x2,x3,mu,a): return (site(x0,x1,x2,x3)*4+mu)*8+a
    def aidx(x0,x1,x2,x3,a): return site(x0,x1,x2,x3)*8+a

    for x0 in range(L):
      for x1 in range(L):
        for x2 in range(L):
          for x3 in range(L):
            for a in range(8):
              col=aidx(x0,x1,x2,x3,a)
              for mu in range(4):
                G[tidx(x0,x1,x2,x3,mu,a), col] += 1
                xm=[x0,x1,x2,x3]
                xm[mu]=(xm[mu]-1)%L
                G[tidx(*xm,mu,a), col] -= 1
    return G

def projector(G):
    U,s,_=np.linalg.svd(G,full_matrices=False)
    mask=s>1e-10
    if np.any(mask):
        Uim=U[:,mask]
        P=np.eye(G.shape[0]) - Uim@Uim.T
    else:
        P=np.eye(G.shape[0])
    return 0.5*(P+P.T)

def apply_P(P, v):
    return P @ v


# ============================================================
# Subspace basis
# ============================================================

def build_subspace(P, n, m=48):
    V=[]
    for _ in range(m):
        v=np.random.randn(n)
        v=apply_P(P,v)
        nrm=np.linalg.norm(v)
        if nrm > 1e-12:
            V.append(v/nrm)
    return np.stack(V)


# ============================================================
# Subspace Hessian via HVP
# ============================================================

def build_Hsub(theta, V, L, beta=1.0):
    m, n = V.shape
    Hsub = np.zeros((m,m))
    for i in range(m):
        Hv = hessian_vec(theta, V[i], L, beta)
        for j in range(m):
            Hsub[j,i] = np.dot(V[j], Hv)
    return 0.5*(Hsub + Hsub.T)


# ============================================================
# Coarse map J and subspace J_sub
# ============================================================

def build_J(L):
    Lc=L//2
    nf=n_params(L); nc=n_params(Lc)
    J=np.zeros((nf,nc))
    def site(x0,x1,x2,x3): return ((x0*L+x1)*L+x2)*L+x3
    for mu in range(4):
        for a in range(8):
            cidx=(0*4+mu)*8+a
            fine_sites=[(0,0,0,0),
                        (1 if mu==0 else 0,
                         1 if mu==1 else 0,
                         1 if mu==2 else 0,
                         1 if mu==3 else 0)]
            for (x0,x1,x2,x3) in fine_sites:
                s=site(x0,x1,x2,x3)
                fidx=(s*4+mu)*8+a
                J[fidx,cidx]=1.0
    return J

def coarse_Hsub(Hsub, Jsub):
    return 0.5*(Jsub.T @ Hsub @ Jsub + (Jsub.T @ Hsub @ Jsub).T)


# ============================================================
# Riccati on small Hessian
# ============================================================

def riccati_small(H, steps=12, eta=0.05):
    I=np.eye(H.shape[0])
    Hn=H.copy()
    for _ in range(steps):
        Hn = Hn @ np.linalg.inv(I + eta*Hn)
    return 0.5*(Hn+Hn.T)

def lambda_min_sub(H, tol=1e-6):
    w=np.linalg.eigvalsh(H)
    w=w[np.abs(w)>tol]
    return float(w.min()) if len(w)>0 else 0.0


# ============================================================
# Matrix-free curvature RG driver
# ============================================================

def curvature_RG_matrixfree(L0=4, a0=1.0, beta0=6.0, steps=3, m=48):
    L = L0
    a = a0
    beta = beta0
    theta0 = np.zeros(n_params(L))

    for step in range(steps):
        print(f"\n=== SCALE a={a:.4f},  L={L} ===")

        G = gauge_matrix(L)
        P = projector(G)

        V = build_subspace(P, n_params(L), m=m)
        Hsub = build_Hsub(theta0, V, L, beta)
        Hren = riccati_small(Hsub)

        CWr = lambda_min_sub(Hren)
        print("CWr =", CWr, "    M_eff =", CWr/a)

        Lc = L//2
        if Lc < 1:
            break

        Pc = projector(gauge_matrix(Lc))
        Vc = build_subspace(Pc, n_params(Lc), m=m)
        Jfull = build_J(L)

        Jsub = np.zeros((m,m))
        for i in range(m):
            JV = Jfull.T @ V[i]
            JV = apply_P(Pc, JV)
            for j in range(m):
                Jsub[j,i] = np.dot(Vc[j], JV)

        Hsub = coarse_Hsub(Hren, Jsub)

        theta0 = np.zeros(n_params(Lc))
        L = Lc
        a = 2*a
        beta = beta  # placeholder


# ============================================================
# RUN
# ============================================================

curvature_RG_matrixfree(L0=4, a0=1.0, beta0=6.0, steps=3, m=32)


=== SCALE a=1.0000,  L=4 ===
CWr = 1.3414994774022868     M_eff = 1.3414994774022868

=== SCALE a=2.0000,  L=2 ===
CWr = 1.174030584184985     M_eff = 0.5870152920924925

=== SCALE a=4.0000,  L=1 ===
CWr = 0.0     M_eff = 0.0


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
jax.config.update("jax_enable_x64", True)

# ============================================================
# SU(3) generators
# ============================================================
def su3_generators():
    lam=[]
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]],complex))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]],complex)/np.sqrt(3))
    return 1j * jnp.stack(lam)/2

_T = su3_generators()
def su3_alg_from_vec(a): return jnp.einsum("...a,aij->...ij",a,_T)
def su3_exp(A): return jax.scipy.linalg.expm(A)

# ============================================================
# Lattice parametrization
# ============================================================
def n_params(L): return L**4*4*8

def theta_to_field(theta,L):
    return theta.reshape(L,L,L,L,4,8)

def build_links(theta,L):
    flat = theta_to_field(theta,L).reshape(-1,8)
    mats = jax.vmap(su3_exp)(jax.vmap(su3_alg_from_vec)(flat))
    return mats.reshape(L,L,L,L,4,3,3)

# ============================================================
# Wilson action & gradient
# ============================================================
def wilson_action(theta,L,beta=1.0):
    U=build_links(theta,L)
    S=0.0
    for mu in range(4):
        for nu in range(mu+1,4):
            U1=U[...,mu,:,:]
            U2=jnp.roll(U[...,nu,:,:],-1,axis=mu)
            U3=jnp.conjugate(jnp.swapaxes(jnp.roll(U[...,mu,:,:],-1,axis=nu),-1,-2))
            U4=jnp.conjugate(jnp.swapaxes(U[...,nu,:,:],-1,-2))
            P=U1@U2@U3@U4
            S+=jnp.sum(1.0 - jnp.real(jnp.einsum("...ii->...",P))/3)
    return beta*S

grad_S = jax.grad(wilson_action)

# ============================================================
# Matrix-free Hessian-vector product (corrected)
# ============================================================
def hessian_vec(theta,v,L,beta=1.0):
    theta_j=jnp.array(theta); v_j=jnp.array(v)
    def g(x): return grad_S(x,L,beta)
    _, Hv = jax.jvp(g, (theta_j,), (v_j,))
    return np.array(Hv)

# ============================================================
# Gauge projection
# ============================================================
def gauge_matrix(L):
    n_t=n_params(L); n_a=L**4*8
    G=np.zeros((n_t,n_a))
    def site(x0,x1,x2,x3): return ((x0*L+x1)*L+x2)*L+x3
    def tidx(x0,x1,x2,x3,mu,a): return (site(x0,x1,x2,x3)*4+mu)*8+a
    def aidx(x0,x1,x2,x3,a): return site(x0,x1,x2,x3)*8+a

    for x0 in range(L):
      for x1 in range(L):
        for x2 in range(L):
          for x3 in range(L):
            for a in range(8):
              col=aidx(x0,x1,x2,x3,a)
              for mu in range(4):
                G[tidx(x0,x1,x2,x3,mu,a),col]+=1
                xm=[x0,x1,x2,x3]; xm[mu]=(xm[mu]-1)%L
                G[tidx(*xm,mu,a),col]-=1
    return G

def projector(G):
    U,s,_=np.linalg.svd(G,full_matrices=False)
    mask=s>1e-10
    if np.any(mask):
        Uim=U[:,mask]
        P=np.eye(G.shape[0]) - Uim@Uim.T
    else:
        P=np.eye(G.shape[0])
    return 0.5*(P+P.T)

def apply_P(P,v): return P@v

# ============================================================
# IR-mode subspace basis
# ============================================================
def build_IR_basis(P, L, m_extra=16):
    """
    IR basis =
      - 32 constant modes (4 directions × 8 generators)
      - + m_extra random smooth modes
    """
    n = n_params(L)
    IR = []

    # constant modes (IR)
    for mu in range(4):
        for a in range(8):
            v=np.zeros(n)
            # set X_{μ,a} = constant over all links in direction μ
            # structure of theta array: flattened index = ((site)*4 + mu)*8 + a
            idx=0
            for x0 in range(L):
              for x1 in range(L):
                for x2 in range(L):
                  for x3 in range(L):
                    s = ((x0*L + x1)*L + x2)*L + x3
                    base = s*32 + mu*8
                    v[base + a] = 1.0
            v = apply_P(P, v)
            nrm=np.linalg.norm(v)
            if nrm>1e-12: IR.append(v/nrm)

    # add some smooth random modes
    for _ in range(m_extra):
        v=np.random.randn(n)
        v = apply_P(P,v)
        v = v / (np.linalg.norm(v)+1e-12)
        IR.append(v)

    # Orthonormalize IR basis
    IR = np.stack(IR)
    Q,_ = np.linalg.qr(IR.T)
    IR = Q.T

    return IR  # shape (m, n)

# ============================================================
# Build subspace Hessian via HVPs
# ============================================================
def build_Hsub(theta, V, L, beta=1.0):
    m, n = V.shape
    Hsub=np.zeros((m,m))
    for i in range(m):
        Hv = hessian_vec(theta, V[i], L, beta)
        for j in range(m):
            Hsub[j,i] = np.dot(V[j], Hv)
    return 0.5*(Hsub+Hsub.T)

# ============================================================
# Coarse map J and subspace J_sub
# ============================================================
def build_J(L):
    Lc=L//2
    nf=n_params(L); nc=n_params(Lc)
    J=np.zeros((nf,nc))
    def site(x0,x1,x2,x3): return ((x0*L+x1)*L+x2)*L+x3
    for mu in range(4):
        for a in range(8):
            cidx=(0*4+mu)*8+a
            fine_sites=[(0,0,0,0),
                        (1 if mu==0 else 0,
                         1 if mu==1 else 0,
                         1 if mu==2 else 0,
                         1 if mu==3 else 0)]
            for (x0,x1,x2,x3) in fine_sites:
                s=site(x0,x1,x2,x3)
                fidx=(s*4+mu)*8 + a
                J[fidx,cidx]=1.0
    return J

def coarse_Hsub(Hsub, Jsub):
    return 0.5*(Jsub.T @ Hsub @ Jsub + (Jsub.T @ Hsub @ Jsub).T)

# ============================================================
# Riccati on small Hessian
# ============================================================
def riccati_small(H,steps=12,eta=0.05):
    I=np.eye(H.shape[0])
    Hn=H.copy()
    for _ in range(steps):
        Hn = Hn @ np.linalg.inv(I + eta*Hn)
    return 0.5*(Hn+Hn.T)

def lambda_min_sub(H,tol=1e-6):
    w=np.linalg.eigvalsh(H)
    w=w[np.abs(w)>tol]
    return float(w.min()) if len(w)>0 else 0.0

# ============================================================
# Matrix-free curvature RG driver for L=8 → 4 → 2 → 1
# ============================================================
def curvature_RG_L8(beta0=6.0, steps=3, m_extra=16):
    L=8
    a=1.0
    beta=beta0
    theta0=np.zeros(n_params(L))

    for step in range(steps):
        print(f"\n=== SCALE a={a:.3f}, L={L} ===")

        P = projector(gauge_matrix(L))
        V = build_IR_basis(P, L, m_extra=m_extra)

        Hsub = build_Hsub(theta0, V, L, beta)
        Hren = riccati_small(Hsub)

        CWr = lambda_min_sub(Hren)
        print("CWr =",CWr,"   M_eff =",CWr/a)

        Lc=L//2
        if Lc < 1:
            break

        Pc = projector(gauge_matrix(Lc))
        Vc = build_IR_basis(Pc, Lc, m_extra=m_extra)

        Jfull = build_J(L)
        m=V.shape[0]
        Jsub=np.zeros((m,m))

        for i in range(m):
            JV = Jfull.T @ V[i]
            JV = apply_P(Pc,JV)
            for j in range(m):
                Jsub[j,i] = np.dot(Vc[j], JV)

        Hsub = coarse_Hsub(Hren,Jsub)

        theta0 = np.zeros(n_params(Lc))
        L = Lc
        a = 2*a
        beta = beta  # placeholder

# ============================================================
# RUN THE L=8 CURVATURE RG
# ============================================================
curvature_RG_L8(beta0=6.0, steps=3, m_extra=16)


=== SCALE a=1.000, L=8 ===


In [1]:
import jax
import jax.numpy as jnp
from jax import vmap
import time

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "gpu")


# ============================================================
# Progress logger
# ============================================================
class Progress:
    def __init__(self):
        self.t0 = time.time()
    def log(self, msg):
        dt = time.time() - self.t0
        print(f"[{dt:8.2f}s] {msg}")


# ============================================================
# SU(3) generators (anti-Hermitian)
# ============================================================
def su3_generators():
    lam=[]
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]],complex))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]],complex)/jnp.sqrt(3))
    return 1j * jnp.stack(lam) / 2

_T = su3_generators()

# algebra map + exp, jitted
alg_from_vec = jax.jit(lambda a: jnp.einsum("...a,aij->...ij", a, _T))
su3_exp      = jax.jit(lambda A: jax.scipy.linalg.expm(A))


# ============================================================
# Lattice structure
# ============================================================
def n_params(L):
    return L**4 * 4 * 8  # sites × directions × generators

def theta_to_field(theta, L):
    return theta.reshape(L, L, L, L, 4, 8)

theta_to_field = jax.jit(theta_to_field, static_argnames=('L',))

def build_links(theta, L):
    flat = theta_to_field(theta, L).reshape(-1, 8)
    A = vmap(alg_from_vec)(flat)
    U = vmap(su3_exp)(A)
    return U.reshape(L, L, L, L, 4, 3, 3)

build_links = jax.jit(build_links, static_argnames=('L',))


# ============================================================
# Wilson action + right-invariant HVP
# ============================================================
def wilson_action(theta, L, beta):
    U = build_links(theta, L)
    S = 0.0
    for mu in range(4):
        U1 = U[..., mu, :, :]
        for nu in range(mu+1, 4):
            U2 = jnp.roll(U[..., nu, :, :], -1, axis=mu)
            U3 = jnp.conjugate(jnp.swapaxes(jnp.roll(U[..., mu, :, :], -1, axis=nu), -2, -1))
            U4 = jnp.conjugate(jnp.swapaxes(U[..., nu, :, :], -2, -1))
            P  = U1 @ U2 @ U3 @ U4
            S += jnp.sum(1 - jnp.real(jnp.trace(P, axis1=-2, axis2=-1))/3.0)
    return beta * S

wilson_action = jax.jit(wilson_action, static_argnames=('L', 'beta'))
grad_S = jax.grad(wilson_action)

def hvp(theta, v, L, beta):
    _, Hv = jax.jvp(lambda x: grad_S(x, L, beta), (theta,), (v,))
    return jnp.real(Hv)  # kill tiny complex noise

hvp = jax.jit(hvp, static_argnames=('L', 'beta'))


# ============================================================
# Gauge stencils G, G^T
# ============================================================
def G_mul(alpha, L):
    # alpha: (L,L,L,L,8) -> v: (L,L,L,L,4,8)
    return jnp.stack([jnp.roll(alpha, -1, axis=mu) - alpha for mu in range(4)], axis=4)

def GT_mul(v, L):
    # v: (L,L,L,L,4,8) -> (L,L,L,L,8)
    res = jnp.zeros(v.shape[:4] + (8,))
    for mu in range(4):
        res += jnp.roll(v[..., mu, :], 1, axis=mu) - v[..., mu, :]
    return res


# ============================================================
# Exact FFT Laplacian solver for (G^T G) alpha = rhs
# ============================================================
def fft_laplace_solve(rhs, L):
    # rhs: (L,L,L,L,8)
    rhs_k = jnp.fft.fftn(rhs, axes=(0,1,2,3))

    k = jnp.arange(L)
    K0, K1, K2, K3 = jnp.meshgrid(k, k, k, k, indexing='ij')

    lam = 4 * (
        jnp.sin(jnp.pi * K0 / L)**2 +
        jnp.sin(jnp.pi * K1 / L)**2 +
        jnp.sin(jnp.pi * K2 / L)**2 +
        jnp.sin(jnp.pi * K3 / L)**2
    )

    lam_safe = jnp.where(lam == 0, 1.0, lam)
    alpha_k = rhs_k / lam_safe[..., None]

    # zero global gauge mode
    alpha_k = alpha_k.at[0,0,0,0,:].set(jnp.zeros(8))

    alpha = jnp.fft.ifftn(alpha_k, axes=(0,1,2,3))
    return jnp.real(alpha)

fft_laplace_solve = jax.jit(fft_laplace_solve, static_argnames=('L',))


# ============================================================
# Projector: P = I - G (G^T G)^-1 G^T
# ============================================================
def build_projector(L):
    def P(v_flat):
        v = v_flat.reshape(L, L, L, L, 4, 8)
        rhs = GT_mul(v, L)
        alpha = fft_laplace_solve(rhs, L)
        return (v - G_mul(alpha, L)).reshape(-1)
    return jax.jit(P)


# ============================================================
# IR basis at L=8 (single basis)
# ============================================================
def build_IR_basis(L, P, extra=16):
    n = n_params(L)
    V = []

    # 1. Constant modes
    for mu in range(4):
        for a in range(8):
            idxs = []
            vals = []
            for x0 in range(L):
                for x1 in range(L):
                    for x2 in range(L):
                        for x3 in range(L):
                            s = ((x0*L + x1)*L + x2)*L + x3
                            idxs.append(s*32 + mu*8 + a)
                            vals.append(1.0)
            v = jnp.zeros(n).at[jnp.array(idxs)].set(jnp.array(vals))
            v = P(v)
            nrm = jnp.linalg.norm(v)
            if nrm > 1e-12:
                V.append(v / nrm)

    # 2. Momentum-1 modes (cos/sin)
    for mu in range(4):
        for a in range(8):
            for trig in ["cos", "sin"]:
                fun = jnp.cos if trig == "cos" else jnp.sin
                idxs = []
                vals = []
                for x0 in range(L):
                    for x1 in range(L):
                        for x2 in range(L):
                            for x3 in range(L):
                                coord = [x0,x1,x2,x3][mu]
                                phase = fun(2*jnp.pi * coord / L)
                                s = ((x0*L + x1)*L + x2)*L + x3
                                idxs.append(s*32 + mu*8 + a)
                                vals.append(phase)
                v = jnp.zeros(n).at[jnp.array(idxs)].set(jnp.array(vals))
                v = P(v)
                nrm = jnp.linalg.norm(v)
                if nrm > 1e-12:
                    V.append(v / nrm)

    # 3. Random block modes
    for k in range(extra):
        v = jax.random.normal(jax.random.PRNGKey(k), (n,))
        v = P(v)
        nrm = jnp.linalg.norm(v)
        if nrm > 1e-12:
            V.append(v / nrm)

    V = jnp.stack(V)
    Q, _ = jnp.linalg.qr(V.T)
    return Q.T  # (m, n)


# ============================================================
# Batched Hessian: H = V (HVP(V))^T
# ============================================================
def build_Hsub(theta, V, L, beta):
    HV = vmap(lambda vi: hvp(theta, vi, L, beta))(V)  # (m, n)
    H = V @ HV.T                                     # (m, m)
    return 0.5 * (H + H.T)


# ============================================================
# Block RG map J: flatten(L) -> flatten(L/2)
# ============================================================
def build_J(L):
    nf = n_params(L)
    Lc = L // 2
    nc = n_params(Lc)
    J = jnp.zeros((nf, nc))

    for mu in range(4):
        for a in range(8):
            cidx = mu*8 + a
            fine = [
                (0,0,0,0),
                (1 if mu==0 else 0,
                 1 if mu==1 else 0,
                 1 if mu==2 else 0,
                 1 if mu==3 else 0),
            ]
            for (x0,x1,x2,x3) in fine:
                s = ((x0*L + x1)*L + x2)*L + x3
                J = J.at[s*32 + mu*8 + a, cidx].set(0.5)
    return J


# ============================================================
# Riccati + curvature
# ============================================================
def riccati_small(H, steps=12, eta=0.05):
    I = jnp.eye(H.shape[0])
    Hn = H
    for _ in range(steps):
        Hn = Hn @ jnp.linalg.inv(I + eta*Hn)
    return 0.5 * (Hn + Hn.T)

def lambda_min(H, tol=1e-6):
    w = jnp.linalg.eigvalsh(H)
    w = w[jnp.abs(w) > tol]
    return float(w.min()) if w.size > 0 else 0.0


# ============================================================
# FINAL RG DRIVER: L = 8 → 4 → 2, then stop
# ============================================================
def curvature_RG_L8(beta=6.0, extra=16):

    prog = Progress()
    L = 8
    a = 1.0
    theta0 = jnp.zeros(n_params(L))

    # Build single IR basis at L=8
    prog.log("Building IR basis at L=8 (single basis)")
    P8 = build_projector(8)
    V = build_IR_basis(8, P8, extra=extra)
    m = V.shape[0]
    prog.log(f"Initial IR basis dimension: {m}")

    scales = []
    curvs  = []

    # We will do at most L=8,4,2; no L=1
    for step in range(3):

        prog.log(f"=== RG SCALE a={a:.3f}  L={L}  step={step} ===")

        # Build projector for this L (for hvp consistency, though hvp only uses theta, not P)
        P_L = build_projector(L)  # not actually used inside hvp, but keep for symmetry

        # Hessian at this scale
        prog.log("Building Hessian")
        Hsub = build_Hsub(theta0, V, L, beta)

        # Riccati flow
        prog.log("Riccati")
        Hren = riccati_small(Hsub)

        # Curvature
        CWr = lambda_min(Hren)
        Meff = CWr / a
        prog.log(f"CWr={CWr}, Meff={Meff}\n")

        scales.append((L, a))
        curvs.append((CWr, Meff))

        # Stop at L = 2 (no L=1 attempt)
        if L <= 2:
            prog.log("Reached L=2: stopping RG to avoid IR/tangent mismatch.")
            break

        # Coarse step: map IR basis down
        prog.log("Coarse RG step")
        Lc = L // 2
        Jf = build_J(L)
        Pc = build_projector(Lc)

        prog.log("Batched coarse mapping of IR basis")
        # Map basis with J, then project gauge
        JV_all = vmap(lambda vi: Jf.T @ vi)(V)    # (m, n_c)
        Vc_raw = vmap(Pc)(JV_all)                # (m, n_c)

        # Orthonormalize coarse basis (keep same m)
        Qc, _ = jnp.linalg.qr(Vc_raw.T)          # (n_c, m)
        V = Qc.T                                 # (m, n_c)

        # Next scale
        theta0 = jnp.zeros(n_params(Lc))
        L = Lc
        a = 2.0 * a

    # Summary
    print("\n========================================")
    print(" CURVATURE RG FLOW SUMMARY (L=8→4→2)")
    print("========================================")
    print(f"{'L':>4} {'a':>8} {'CWr':>20} {'M_eff':>20}")
    for (L,a),(cwr,meff) in zip(scales, curvs):
        print(f"{L:>4} {a:>8.3f} {cwr:>20.12e} {meff:>20.12e}")
    print("========================================\n")


# ============================================================
# RUN
# ============================================================
curvature_RG_L8(beta=6.0, extra=16)


[    0.06s] Building IR basis at L=8 (single basis)
[   90.31s] Initial IR basis dimension: 48
[   90.31s] === RG SCALE a=1.000  L=8  step=0 ===
[   90.31s] Building Hessian


/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


[  106.28s] Riccati
[  107.34s] CWr=1.3773161794003692, Meff=1.3773161794003692

[  107.34s] Coarse RG step
[  111.93s] Batched coarse mapping of IR basis


XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 8606711808 bytes.

In [2]:
    return scales, curvs


SyntaxError: 'return' outside function (ipython-input-707049415.py, line 1)